# 03 - Analyse des resultats

In [ ]:
import sys
sys.path.append('..')
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from pathlib import Path
from src.data.tokenizer import load_config
from src.evaluation.metrics import compute_metrics, plot_confusion_matrix, plot_roc_curve, apply_thresholds
config = load_config('../configs/config.yaml')

## Charger le modele CNN entraine

In [ ]:
classifier_path = Path('../saved_models/classifier/best_model.h5')
if classifier_path.exists():
    model = tf.keras.models.load_model(str(classifier_path))
    print('Model loaded')
    model.summary()
else:
    print('Run train_classifier.py first.')

## Evaluation sur le test set

In [ ]:
from src.training.train_classifier import load_images_as_tensors
from sklearn.model_selection import train_test_split
X, y = load_images_as_tensors('../data/images', config)
_, X_temp, _, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)
print(f'Test samples: {X_test.shape[0]}')

In [ ]:
if classifier_path.exists():
    y_scores = model.predict(X_test).flatten()
    y_pred, _ = apply_thresholds(y_scores, config)
    metrics = compute_metrics(y_test, y_pred, y_scores)
    for k, v in metrics.items():
        print(f'  {k:10s}: {v:.4f}')

## Matrice de confusion

In [ ]:
if classifier_path.exists():
    plot_confusion_matrix(y_test, y_pred, '../results/confusion_matrix.png')
    from IPython.display import Image as IPImage
    display(IPImage('../results/confusion_matrix.png'))

## Courbe ROC

In [ ]:
if classifier_path.exists():
    plot_roc_curve(y_test, y_scores, '../results/roc_curve.png')
    from IPython.display import Image as IPImage
    display(IPImage('../results/roc_curve.png'))

## Distribution des scores

In [ ]:
if classifier_path.exists():
    fig, ax = plt.subplots(figsize=(8,4))
    ax.hist(y_scores[y_test==0], bins=50, alpha=0.7, color='steelblue', label='Benign')
    ax.hist(y_scores[y_test==1], bins=50, alpha=0.7, color='crimson', label='Malware')
    ax.axvline(config['thresholds']['block'], color='red', linestyle='--', label='Block')
    ax.axvline(config['thresholds']['quarantine'], color='orange', linestyle='--', label='Quarantine')
    ax.set_xlabel('Malware probability score')
    ax.set_ylabel('Count')
    ax.set_title('Score distribution')
    ax.legend()
    plt.tight_layout()
    plt.show()